# Import Libraries

In [1]:
import os

import pandas as pd
import numpy as np
import nltk
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder


In [2]:
import sys, time, math, re, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python {sys.version.split()} | PyTorch {torch.__version__} | Device: {device}")

Python ['3.10.10', '(tags/v3.10.10:aad5f6a,', 'Feb', '7', '2023,', '17:20:36)', '[MSC', 'v.1929', '64', 'bit', '(AMD64)]'] | PyTorch 2.12.0+cu126 | Device: cuda


In [3]:
RANDOM_SEED = 42

# Read Data

In [4]:
path = "../../data/gold_labeled_data"
train = pd.read_excel(os.path.join(path, "train.xlsx"))
test = pd.read_excel(os.path.join(path, "test.xlsx"))
print("Number of rows and columns in the train data set:", train.shape)
print("Number of rows and columns in the test data set:", test.shape)
train.head()

Number of rows and columns in the train data set: (2714, 6)
Number of rows and columns in the test data set: (480, 6)


,app_name,text,rating,thumbs_up_count,label_gold,summary_gold
0,Avito,программа Авито очень помогает в продаже б.у т...,5,0,user_experience,Положительный опыт использования приложения
1,VK,Скачала и пользовалась с 2015 года. Новый отзы...,1,24,user_experience,Недовольство модерацией и поддержкой
2,Yandex Maps,Сотрудники Яндекс Поддержки продались и сотруд...,1,5,user_experience,Мешает навязчивая реклама
3,Telegram,"прошу, сделайте так чтоб подарки можно было пр...",5,0,feature_request,Разрешить продажу подарков в любое время
4,Yandex Maps,настройте приложение опять заедает и вылетает,1,3,bug_report,Приложение вылетает


# Label encoding

In [5]:
label_dict = {
    "bug_report": 0,
    "feature_request": 1,
    "rating": 2,
    "user_experience": 3
}

In [6]:
train["label_num"] = train["label_gold"].map(label_dict).astype("int16")
test["label_num"] = test["label_gold"].map(label_dict).astype("int16")

In [7]:
train["label_num"].value_counts()

label_num
3    1289
0     705
2     575
1     145
Name: count, dtype: int64

In [8]:
from gensim.utils import simple_preprocess

train['text'].head().apply(simple_preprocess)

0    [программа, авито, очень, помогает, продаже, т...
1    [скачала, пользовалась, года, новый, отзыв, пи...
2    [сотрудники, яндекс, поддержки, продались, сот...
3    [прошу, сделайте, так, чтоб, подарки, можно, б...
4    [настройте, приложение, опять, заедает, вылетает]
Name: text, dtype: object

## Новый вариант препроцессинга

In [ ]:
import re
import numpy as np
import pandas as pd

from razdel import tokenize
import pymorphy2

from sklearn.feature_extraction.text import TfidfVectorizer
import compress_fasttext
from nltk.corpus import stopwords


# -------------------------
# 1) Морфология и токенизация
# -------------------------
morph = pymorphy2.MorphAnalyzer()

# при желании можно добавить свои стоп-слова
raw_stops = stopwords.words('russian')

# Лемматизируем сам список стоп-слов
stop_words = set(
    morph.parse(w)[0].normal_form
    for w in raw_stops
)

def normalize_ru(word: str) -> str:
    """
    Лемматизация через PyMorphy2.
    """
    word = word.lower().replace("ё", "е")
    return morph.parse(word)[0].normal_form


def text_preproc_ru(text: str) -> str:
    """
    Токенизация Razdel -> очистка -> лемматизация -> строка токенов.
    """
    text = "" if text is None else str(text)
    out = []

    for t in tokenize(text):
        w = t.text.lower().replace("ё", "е")

        # оставляем только слова/смешанные токены с кириллицей или латиницей
        if not re.search(r"[а-яa-z]", w, flags=re.I):
            continue

        if w in stop_words:
            continue

        out.append(normalize_ru(w))

    return " ".join(out)



In [10]:
train['tokens'] = train['text'].apply(text_preproc_ru)
test['tokens'] = test['text'].apply(text_preproc_ru)

In [11]:
train['tokens'].apply(str.split).apply(list.__len__).max()

np.int64(66)

In [12]:
test['tokens'].apply(str.split).apply(list.__len__).max()

np.int64(57)

In [13]:
from collections import Counter

counter = Counter()

for text in train['tokens']:
    counter.update(text.split())

MIN_FREQ = 1
MAX_VOCAB = 30_000

itos = ["<PAD>", "<UNK>"] + [w for w, c in counter.most_common(MAX_VOCAB) if c >= MIN_FREQ]

In [14]:
stoi = {w: i for i, w in enumerate(itos)}

PAD_IDX = stoi["<PAD>"]   # 0 — паддинг, не несёт информации
UNK_IDX = stoi["<UNK>"]   # 1 — неизвестное слово
VOCAB_SIZE = len(itos)

print(f"Vocab size : {VOCAB_SIZE}")
print(f"OOV rate   : {sum(1 for w in counter if w not in stoi) / len(counter) * 100:.1f}%")

Vocab size : 5490
OOV rate   : 0.0%


In [15]:
MAX_LEN = train['tokens'].apply(str.split).apply(list.__len__).max()  # обрезаем/паддим все тексты до одной длины

def encode(text: str, max_len=MAX_LEN) -> list[int]:
    tokens = text.split()[:max_len]
    ids = [stoi.get(t, UNK_IDX) for t in tokens]
    ids += [PAD_IDX] * (max_len - len(ids))  # правый паддинг
    return ids


train["tokens_encoded"] = train["tokens"].map(encode)
test["tokens_encoded"] = test["tokens"].map(encode)

In [16]:
decoded = [[itos[j] for j in i] for i in train["tokens_encoded"]
        #    if i not in (PAD_IDX, UNK_IDX)
           ]
# print("Original :", train_ds["text"][:80])
# print("Decoded  :", " ".join(decoded[:12]))

In [17]:
train

,app_name,text,rating,thumbs_up_count,label_gold,summary_gold,label_num,tokens,tokens_encoded
0,Avito,программа Авито очень помогает в продаже б.у т...,5,0,user_experience,Положительный опыт использования приложения,3,программа авить очень помогать продажа б тоаюв...,"[455, 26, 7, 134, 211, 629, 2494, 2495, 630, 2..."
1,VK,Скачала и пользовалась с 2015 года. Новый отзы...,1,24,user_experience,Недовольство модерацией и поддержкой,3,скачать пользоваться год новый отзыв писать на...,"[160, 27, 53, 40, 43, 22, 696, 2496, 631, 2497..."
2,Yandex Maps,Сотрудники Яндекс Поддержки продались и сотруд...,1,5,user_experience,Мешает навязчивая реклама,3,сотрудник яндекс поддержка продаться сотруднич...,"[698, 44, 33, 1765, 1370, 342, 44, 10, 1370, 8..."
3,Telegram,"прошу, сделайте так чтоб подарки можно было пр...",5,0,feature_request,Разрешить продажу подарков в любое время,1,просить сделать подарок быть продать угодный день,"[175, 17, 869, 6, 285, 456, 59, 0, 0, 0, 0, 0,..."
4,Yandex Maps,настройте приложение опять заедает и вылетает,1,3,bug_report,Приложение вылетает,0,настроить приложение заедать вылетать,"[527, 2, 2507, 262, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
...,...,...,...,...,...,...,...,...,...
2709,Avito,сделка состоялась без проблем👍,5,0,user_experience,Положительный опыт использования приложения,3,сделка состояться проблем👍,"[636, 987, 5485, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
2710,Avito,"Заставили меня обновить приложение,хоть и не х...",1,1,bug_report,Приложение не открывается или не запускается,0,заставить я обновить приложение хотеть знать о...,"[1150, 5, 178, 2, 45, 154, 14, 5486, 116, 116,..."
2711,VK,"максимально печальное приложение, хуже просто ...",1,5,bug_report,Недовольство модерацией и поддержкой,0,максимально печальный приложение плохой просто...,"[546, 5487, 2, 32, 11, 309, 509, 16, 682, 136,..."
2712,Yandex Go,"нету другого сервиса , таксисты не приезжают н...",1,0,user_experience,Недовольство платными функциями,3,нету другой сервис таксист приезжать локация п...,"[309, 65, 83, 638, 1146, 811, 5488, 1701, 495,..."


## Добавляем в DataLoader данные rating и thumbs

In [18]:
def make_meta_features(df):
    meta = df[["rating", "thumbs_up_count"]].copy()

    # На всякий случай заполняем пропуски
    meta["rating"] = meta["rating"].fillna(meta["rating"].median())
    meta["thumbs_up_count"] = meta["thumbs_up_count"].fillna(0)

    # thumbs_up_count сильно перекошен, поэтому логарифмируем
    meta["thumbs_up_count"] = np.log1p(meta["thumbs_up_count"])

    return meta.values.astype("float32")

In [19]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler


def make_loaders(X_ids, X_meta, y, val_size=0.2, batch_size=128):
    X_tr, X_val, meta_tr, meta_val, y_tr, y_val = train_test_split(
        X_ids,
        X_meta,
        y,
        test_size=val_size,
        stratify=y,
        random_state=RANDOM_SEED
    )

    # scaler обучаем только на train, чтобы не было утечки из validation
    meta_scaler = StandardScaler()
    meta_tr = meta_scaler.fit_transform(meta_tr).astype("float32")
    meta_val = meta_scaler.transform(meta_val).astype("float32")

    def to_loader(X, meta, y, shuffle):
        ds = TensorDataset(
            torch.from_numpy(X).long(),
            torch.from_numpy(meta).float(),
            torch.from_numpy(y).long()
        )
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

    train_loader = to_loader(X_tr, meta_tr, y_tr, True)
    val_loader = to_loader(X_val, meta_val, y_val, False)

    return train_loader, val_loader, meta_scaler


X_ids = np.array([i for i in train["tokens_encoded"]])
X_meta = make_meta_features(train)
y_ids = np.array([i for i in train["label_num"]])

train_loader, val_loader, meta_scaler = make_loaders(X_ids, X_meta, y_ids)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 17, Val batches: 5


## Функция валидации

In [20]:
def evaluate_model(model, loader):
    model.eval()
    crit = nn.CrossEntropyLoss()
    total_loss, all_y, all_p = 0.0, [], []

    with torch.no_grad():
        for xb, meta, yb in loader:
            xb = xb.to(device)
            meta = meta.to(device)
            yb = yb.to(device)

            logits = model(xb, meta)

            total_loss += crit(logits, yb).item() * xb.size(0)
            all_y.extend(yb.cpu().numpy())
            all_p.extend(logits.argmax(1).cpu().numpy())

    n = len(all_y)

    return {
        "loss": total_loss / n,
        "acc": accuracy_score(all_y, all_p),
        "f1": f1_score(all_y, all_p, average="macro"),
    }

## Функция тренировки

In [21]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

weights = compute_class_weight('balanced', classes=np.unique(y_ids), y=y_ids)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
crit = nn.CrossEntropyLoss(weight=class_weights)

In [22]:
# import wandb


def train_model(model, train_loader, val_loader,
          run_name="run", epochs=3, lr=1e-3, wd=1e-4, crit_=None, early_stopping=True):

    model = model.to(device)
    opt  = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', patience=3, factor=0.5)

    if not crit_:
        crit = nn.CrossEntropyLoss()
    else:
        crit = crit_


    patience, no_improve = 5, 0
    # wandb.init(project="nlp-seminar-task-2", name=run_name,
    #            config=dict(epochs=epochs, lr=lr, wd=wd,
    #                        model=type(model).__name__), reinit=True)

    best_f1, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum, n = 0.0, 0

        for xb, meta, yb in train_loader:
            xb = xb.to(device)
            meta = meta.to(device)
            yb = yb.to(device)

            opt.zero_grad()

            logits = model(xb, meta)
            loss = crit(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            loss_sum += loss.item() * xb.size(0)
            n += xb.size(0)

        m = evaluate_model(model, val_loader)
        # wandb.log({"epoch": epoch,
        #            "train_loss": loss_sum / n,
        #            "val_loss": m["loss"],
        #            "val_acc":  m["acc"],
        #            "val_f1":   m["f1"]})

        print(f"[{epoch}/{epochs}] train_loss={(loss_sum/n):.4f}  val_loss={m['loss']:.4f}  "
              f"val_acc={m['acc']:.4f}  val_f1={m['f1']:.4f}")


        if m["f1"] > best_f1:
            best_f1 = m["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            if early_stopping:
                no_improve += 1
                if no_improve >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break

        scheduler.step(m["f1"])
    model.load_state_dict(best_state)
    # wandb.summary["best_val_f1"] = best_f1
    # wandb.finish()
    return model

## Создаем CNN

In [23]:
class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, n_filters=100,
                 filter_sizes=(3, 4, 5), num_classes=4,
                 dropout=0.5, pad_idx=0, meta_dim=2):
        super().__init__()
        self.meta_dim = meta_dim
        self.emb   = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.emb_dropout = nn.Dropout(0.3)

        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(emb_dim, n_filters, k),
                nn.BatchNorm1d(n_filters),
            ) for k in filter_sizes
        ])

        cnn_features_dim = n_filters * len(filter_sizes)
        total_features_dim = cnn_features_dim + meta_dim
        
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(total_features_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, ids, meta):
        # ids: (B, L)
        # meta: (B, meta_dim)

        x = self.emb_dropout(self.emb(ids)).permute(0, 2, 1)
        # x: (B, D, L)

        pooled = []

        for conv in self.convs:
            out = F.relu(conv(x))
            pooled.append(F.max_pool1d(out, kernel_size=out.size(2)).squeeze(2))

        text_features = torch.cat(pooled, dim=1)
        # text_features: (B, n_filters * len(filter_sizes))

        features = torch.cat([text_features, meta], dim=1)
        # features: (B, text_features + rating/thumbs)

        return self.head(features)

In [24]:
NUM_CLASSES = 4

wv_model = compress_fasttext.models.CompressedFastTextKeyedVectors.load(
    "../../models/geowac_tokens_sg_300_5_2020-400K-100K-300.bin"
)
W2V_DIM = wv_model.vector_size

# Создаём матрицу (vocab_size, w2v_dim), OOV — случайный вектор
matrix = np.random.normal(scale=0.01, size=(VOCAB_SIZE, W2V_DIM)).astype("float32")

hits = 0
for i, word in enumerate(itos):
    if word in wv_model:
        matrix[i] = wv_model[word]
        hits += 1

In [27]:
# cnn = CNNClassifier(VOCAB_SIZE, pad_idx=PAD_IDX)

# sample = torch.from_numpy(X_ids[:4]).long()
# with torch.no_grad():
#     out = cnn(sample)

# print(f"input:  {sample.shape}   →  (batch=4, MAX_LEN={MAX_LEN})")
# print(f"output: {out.shape}      →  (batch=4, num_classes={NUM_CLASSES})")

In [28]:
# Создаём матрицу (vocab_size, w2v_dim), OOV — случайный вектор
matrix = np.random.normal(scale=0.01, size=(VOCAB_SIZE, W2V_DIM)).astype("float32")

hits = 0
for i, word in enumerate(itos):
    if word in wv_model:
        matrix[i] = wv_model[word]
        hits += 1

matrix.shape

(5490, 300)

In [29]:
cnn_with_embeds = CNNClassifier(
    VOCAB_SIZE,
    n_filters=200,
    filter_sizes=(2, 3, 4, 5, 6),
    emb_dim=W2V_DIM,
    pad_idx=PAD_IDX,
    num_classes=NUM_CLASSES,
    meta_dim=2
)

with torch.no_grad():
    cnn_with_embeds.emb.weight.copy_(torch.from_numpy(matrix))

cnn_with_embeds.emb.weight.requires_grad_(False)

Parameter containing:
tensor([[ 0.0513,  0.1763, -0.0251,  ..., -0.1866, -0.0427,  0.1037],
        [-0.0992, -0.0263,  0.1041,  ..., -0.1095,  0.0899,  0.1924],
        [ 0.0689,  0.2604,  0.0659,  ...,  0.1989, -0.1366,  0.0435],
        ...,
        [-0.4055,  0.1190, -0.1744,  ...,  0.2519,  0.1210, -0.0697],
        [-0.4991, -0.0197, -0.0185,  ..., -0.4089,  0.1262, -0.1636],
        [-0.0374,  0.1475,  0.5497,  ...,  0.0103,  0.3937, -0.0324]])

In [30]:
cnn_with_embeds = train_model(
    cnn_with_embeds,
    train_loader,
    val_loader,
    run_name="cnn-fasttext-with-rating-thumbs",
    epochs=50,
    crit_=crit,
    early_stopping=False
)

[1/50] train_loss=1.5339  val_loss=1.2100  val_acc=0.4641  val_f1=0.4320
[2/50] train_loss=0.9734  val_loss=0.8972  val_acc=0.6114  val_f1=0.4854
[3/50] train_loss=0.7354  val_loss=0.8450  val_acc=0.6077  val_f1=0.5763
[4/50] train_loss=0.6480  val_loss=0.7757  val_acc=0.6446  val_f1=0.6216
[5/50] train_loss=0.5522  val_loss=0.7118  val_acc=0.7164  val_f1=0.6878
[6/50] train_loss=0.4880  val_loss=0.7167  val_acc=0.7053  val_f1=0.6933
[7/50] train_loss=0.4798  val_loss=0.6512  val_acc=0.7440  val_f1=0.7111
[8/50] train_loss=0.4031  val_loss=0.6569  val_acc=0.7459  val_f1=0.7283
[9/50] train_loss=0.3861  val_loss=0.6575  val_acc=0.7459  val_f1=0.7267
[10/50] train_loss=0.3406  val_loss=0.6565  val_acc=0.7422  val_f1=0.7287
[11/50] train_loss=0.3052  val_loss=0.6317  val_acc=0.7514  val_f1=0.7130
[12/50] train_loss=0.3644  val_loss=0.6339  val_acc=0.7569  val_f1=0.7313
[13/50] train_loss=0.3183  val_loss=0.6684  val_acc=0.7514  val_f1=0.7327
[14/50] train_loss=0.3041  val_loss=0.7206  val

In [31]:
m = evaluate_model(cnn_with_embeds, val_loader)
print(f"CNN — val_f1: {m['f1']:.4f}")

CNN — val_f1: 0.7327


# Смотрим как отработало на тестовой выборке

In [33]:
def evaluate_model_for_upload(model, loader):
    model.eval()
    all_p = []

    with torch.no_grad():
        for xb, meta in loader:
            xb = xb.to(device)
            meta = meta.to(device)

            logits = model(xb, meta)
            all_p.extend(logits.argmax(1).cpu().numpy())

    return all_p

In [36]:
X_test_ids = np.array([i for i in test["tokens_encoded"]])
X_test_meta = make_meta_features(test)

X_test_meta = meta_scaler.transform(X_test_meta).astype("float32")

X_test_ids_tensor = torch.from_numpy(X_test_ids).long()
X_test_meta_tensor = torch.from_numpy(X_test_meta).float()

ds_for_load = TensorDataset(
    X_test_ids_tensor,
    X_test_meta_tensor
)

test_loader_for_upload = DataLoader(
    ds_for_load,
    batch_size=128,
    shuffle=False
)

In [37]:
cnn_with_embeds_test = cnn_with_embeds.to(device)
preds = evaluate_model_for_upload(cnn_with_embeds_test, test_loader_for_upload)

In [38]:
# preds = preds.argmax(1).cpu().numpy()
preds

[np.int64(0),
 np.int64(3),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(3),
 np.int64(2),
 np.int64(3),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(0),
 np.int64(3),
 np.int64(0),
 np.int64(3),
 np.int64(3),
 np.int64(2),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(3),
 np.int64(3),
 np.int64(1),
 np.int64(3),
 np.int64(3),
 np.int64(2),
 np.int64(2),
 np.int64(3),
 np.int64(3),
 np.int64(2),
 np.int64(0),
 np.int64(0),
 np.int64(3),
 np.int64(1),
 np.int64(3),
 np.int64(2),
 np.int64(2),
 np.int64(3),
 np.int64(1),
 np.int64(3),
 np.int64(0),
 np.int64(3),
 np.int64(2),
 np.int64(0),
 np.int64(3),
 np.int64(0),
 np.int64(3),
 np.int64(3),
 np.int64(3),
 np.int64(3),
 np.int64(0),
 np.int64(3),
 np.int64(1),
 np.int64(0),
 np.int64(2),
 np.int64(3),
 np.int64(0),
 np.int64(3),
 np.int64(3),
 np.int64(0),
 np.int64(3),
 np.int64(3),
 np.int64(0),
 np.int64(2),
 np.int64(2),
 np.in

In [42]:
print(classification_report(test["label_num"], preds))

              precision    recall  f1-score   support

           0       0.62      0.90      0.73       125
           1       0.56      0.73      0.63        26
           2       0.87      0.86      0.87       101
           3       0.87      0.63      0.73       228

    accuracy                           0.75       480
   macro avg       0.73      0.78      0.74       480
weighted avg       0.79      0.75      0.75       480



## Запускаем сетку поиска лучших конфигураций

In [65]:
def make_cnn_with_emb(pretrained=None, freeze=False):
    """
    pretrained: numpy матрица (vocab_size, emb_dim) или None
    freeze: не обновлять эмбеддинги во время обучения
    """
    emb_dim = pretrained.shape[1] if pretrained is not None else 128

    model = CNNClassifier(VOCAB_SIZE, emb_dim=emb_dim,
                          n_filters=100, filter_sizes=(3,4,5),
                          num_classes=NUM_CLASSES, pad_idx=PAD_IDX)

    if pretrained is not None:
        with torch.no_grad():
            model.emb.weight.copy_(torch.from_numpy(pretrained))

    if freeze:
        model.emb.weight.requires_grad_(False)

    return model

In [67]:
results = {}
for strategy, pretrained, freeze in [
    ("random-init",    None,   False),
    ("glove-freeze",   matrix, True),
    ("glove-finetune", matrix, False),
]:
    model = make_cnn_with_emb(pretrained, freeze)
    model = train_model(model, train_loader, val_loader,
                  run_name=f"cnn-{strategy}", epochs=50, crit_=crit)
    m = evaluate_model(model, val_loader)
    results[f"cnn-{strategy}"] = m["f1"]
    print(f"[cnn-{strategy}] val_f1 = {m['f1']:.4f}")

print("\n--- Full leaderboard ---")
for name, f1 in sorted(results.items(), key=lambda x: x[1]):
    print(f"  {f1:.4f}  {name}")

[1/50] train_loss=1.5459  val_loss=1.2233  val_acc=0.4494  val_f1=0.3596
[2/50] train_loss=1.2527  val_loss=1.1553  val_acc=0.4475  val_f1=0.4444
[3/50] train_loss=1.0750  val_loss=1.0744  val_acc=0.4622  val_f1=0.4454
[4/50] train_loss=0.9587  val_loss=0.9386  val_acc=0.5875  val_f1=0.5491
[5/50] train_loss=0.8914  val_loss=1.0616  val_acc=0.4751  val_f1=0.4715
[6/50] train_loss=0.7852  val_loss=0.9676  val_acc=0.5322  val_f1=0.5201
[7/50] train_loss=0.7639  val_loss=0.9310  val_acc=0.5543  val_f1=0.5386
[8/50] train_loss=0.6864  val_loss=0.8302  val_acc=0.6077  val_f1=0.5879
[9/50] train_loss=0.7431  val_loss=0.8248  val_acc=0.6022  val_f1=0.5790
[10/50] train_loss=0.6037  val_loss=0.7951  val_acc=0.6483  val_f1=0.6109
[11/50] train_loss=0.5862  val_loss=0.8137  val_acc=0.6206  val_f1=0.5968
[12/50] train_loss=0.5997  val_loss=0.8331  val_acc=0.6114  val_f1=0.5918
[13/50] train_loss=0.5316  val_loss=0.8849  val_acc=0.5893  val_f1=0.5662
[14/50] train_loss=0.5402  val_loss=0.8616  val

In [68]:
def run_cnn(run_name, emb_dim=128, n_filters=100,
            filter_sizes=(3,4,5), dropout=0.5, epochs=30):
    pretrained = matrix
    model = CNNClassifier(VOCAB_SIZE, emb_dim=pretrained.shape[1],
                          n_filters=n_filters, filter_sizes=filter_sizes,
                          num_classes=NUM_CLASSES, pad_idx=PAD_IDX, dropout=dropout,)

    if pretrained is not None:
        with torch.no_grad():
            model.emb.weight.copy_(torch.from_numpy(pretrained))

    if freeze:
        model.emb.weight.requires_grad_(False)
    model = train_model(model, train_loader, val_loader,
                  run_name=run_name, epochs=epochs, crit_=crit)

    m = evaluate_model(model, val_loader)
    print(f"[{run_name}] val_f1 = {m['f1']:.4f}")
    return model, m["f1"]

In [69]:
results = {}

# Варианты: меньше фильтров, больше фильтров, другие окна, больше дропаут
experiments = {
    "cnn-baseline":     dict(n_filters=100, filter_sizes=(3,4,5), dropout=0.5),
    "cnn-more-filters": dict(n_filters=200, filter_sizes=(3,4,5, 6), dropout=0.5),
    "cnn-wide-windows": dict(n_filters=100, filter_sizes=(4,5,6), dropout=0.5),
    "cnn-hi-dropout":   dict(n_filters=100, filter_sizes=(3,4,5), dropout=0.7),
}

for name, cfg in experiments.items():
    _, f1 = run_cnn(name, **cfg)
    results[name] = f1

[1/30] train_loss=1.3190  val_loss=1.1963  val_acc=0.6243  val_f1=0.5439
[2/30] train_loss=0.9510  val_loss=0.9690  val_acc=0.6188  val_f1=0.5365
[3/30] train_loss=0.7771  val_loss=0.8236  val_acc=0.6648  val_f1=0.6575
[4/30] train_loss=0.6743  val_loss=0.7887  val_acc=0.6593  val_f1=0.6293
[5/30] train_loss=0.5749  val_loss=0.7253  val_acc=0.7053  val_f1=0.7006
[6/30] train_loss=0.5559  val_loss=0.7732  val_acc=0.6888  val_f1=0.6758
[7/30] train_loss=0.4762  val_loss=0.7847  val_acc=0.6722  val_f1=0.6456
[8/30] train_loss=0.4088  val_loss=0.6816  val_acc=0.7403  val_f1=0.7279
[9/30] train_loss=0.3487  val_loss=0.6414  val_acc=0.7477  val_f1=0.7235
[10/30] train_loss=0.2945  val_loss=0.6817  val_acc=0.7551  val_f1=0.7316
[11/30] train_loss=0.2604  val_loss=0.7136  val_acc=0.7735  val_f1=0.7547
[12/30] train_loss=0.2161  val_loss=0.7235  val_acc=0.7551  val_f1=0.7341
[13/30] train_loss=0.2264  val_loss=0.6988  val_acc=0.7514  val_f1=0.7302
[14/30] train_loss=0.1959  val_loss=0.7362  val

In [70]:
print("\n--- Full leaderboard ---")
for name, f1 in sorted(results.items(), key=lambda x: x[1]):
    print(f"  {f1:.4f}  {name}")


--- Full leaderboard ---
  0.7294  cnn-wide-windows
  0.7348  cnn-more-filters
  0.7461  cnn-hi-dropout
  0.7547  cnn-baseline
